In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("/kaggle/working/co-tracker").exists():
    subprocess.run(["git", "clone", "https://github.com/facebookresearch/co-tracker.git", "/kaggle/working/co-tracker"], check=True)

os.chdir("/kaggle/working/co-tracker")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

checkpoint = Path("checkpoints/scaled_online.pth")
checkpoint.parent.mkdir(parents=True, exist_ok=True)

if not checkpoint.exists():
    subprocess.run(
        [
            "curl",
            "-L",
            "https://huggingface.co/facebook/cotracker3/resolve/main/scaled_online.pth",
            "-o",
            str(checkpoint),
        ],
        check=True,
    )

subprocess.run(["ls", "-lh", str(checkpoint)], check=True)

In [ ]:
import cv2
import torch
import numpy as np
from collections import deque
from pathlib import Path

from cotracker.predictor import CoTrackerOnlinePredictor

INPUT_ROOT = Path("/kaggle/input/datasets/dharun235/lateral-insect-view/Lateral view insect video")
OUTPUT_ROOT = Path("/kaggle/working/dense-track")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT = "checkpoints/scaled_online.pth"
VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov", ".mkv")
SKIP_EXISTING = True

width = 640
height = 360
grid_size = 30

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)


def clear_device_cache():
    if device == "cuda":
        torch.cuda.empty_cache()


def process_video(video_path):
    video_path = Path(video_path)
    output_stem = video_path.stem
    tracks_file = OUTPUT_ROOT / f"{output_stem}_tracks.pt"
    vis_video_mp4 = OUTPUT_ROOT / f"{output_stem}_tracking.mp4"

    if SKIP_EXISTING and tracks_file.exists() and vis_video_mp4.exists():
        print(f"Skipping existing outputs: {video_path.name}")
        return tracks_file, vis_video_mp4

    print(f"\nProcessing: {video_path.name}")

    model = CoTrackerOnlinePredictor(checkpoint=CHECKPOINT).to(device)
    model.eval()

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 30

    tracks_all = []
    visibility_all = []
    step = model.step
    print("CoTracker step:", step)

    buffer = deque(maxlen=step * 2)
    frame_index = 0
    first_step = True
    first_prediction = True

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (width, height))

        buffer.append(frame)
        frame_index += 1

        if len(buffer) < step * 2 or frame_index % step != 0:
            continue

        video = (
            torch.from_numpy(np.stack(buffer))
            .permute(0, 3, 1, 2)
            .unsqueeze(0)
            .float()
            .to(device)
        )

        with torch.no_grad():
            if first_step:
                model(video_chunk=video, is_first_step=True, grid_size=grid_size)
                first_step = False
                print("Initialized tracking")
                del video
                clear_device_cache()
                continue

            pred_tracks, pred_visibility = model(video_chunk=video)

        if pred_tracks is not None:
            pred_tracks = pred_tracks.cpu()
            pred_visibility = pred_visibility.cpu()

            if first_prediction:
                tracks_all.append(pred_tracks)
                visibility_all.append(pred_visibility)
                first_prediction = False
            else:
                tracks_all.append(pred_tracks[:, -step:])
                visibility_all.append(pred_visibility[:, -step:])

            print("Tracked:", pred_tracks.shape)
            del pred_tracks, pred_visibility

        del video
        clear_device_cache()

    cap.release()

    if not tracks_all:
        print(f"No tracks produced: {video_path.name}")
        del model
        clear_device_cache()
        return None

    tracks = torch.cat(tracks_all, dim=1)
    visibility = torch.cat(visibility_all, dim=1)

    torch.save(
        {
            "video": video_path.name,
            "width": width,
            "height": height,
            "fps": fps,
            "grid_size": grid_size,
            "tracks": tracks,
            "visibility": visibility,
        },
        tracks_file,
    )

    print("Saved tracks:", tracks_file)
    print("Tracks shape:", tracks.shape)

    cap = cv2.VideoCapture(str(video_path))
    writer = cv2.VideoWriter(
        str(vis_video_mp4),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create MP4 writer: {vis_video_mp4}")

    tracks_np = tracks[0].numpy()
    vis_np = visibility[0].numpy()
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret or frame_id >= len(tracks_np):
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (width, height))

        points = tracks_np[frame_id]
        visible = vis_np[frame_id]

        for p in range(len(points)):
            if visible[p]:
                x, y = points[p]
                cv2.circle(frame, (int(x), int(y)), 2, (0, 255, 0), -1)

        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        frame_id += 1

    cap.release()
    writer.release()
    print("Saved visualization:", vis_video_mp4)

    del model, tracks, visibility
    clear_device_cache()

    return tracks_file, vis_video_mp4


video_files = sorted(p for p in INPUT_ROOT.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)

print(f"Found {len(video_files)} videos")
for video_path in video_files:
    process_video(video_path)

print("Done")